# Job Description Analyzer
### Built by Muazzam Hussain — Agentic AI & RAG Systems Builder

This tool:
1. Scrapes a job posting URL
2. Analyzes it against your profile
3. Returns a full breakdown: required skills, nice-to-haves, red flags, fit score, tailored pitch, and interview prep questions

**Stack:** Ollama (llama3.2) + BeautifulSoup + OpenAI-compatible SDK

In [ ]:
# ── Imports 
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
# ── Connect to Ollama 
client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'
)

MODEL = "llama3.2"

print("✓ Connected to Ollama")

## Your Profile
This is what gets passed to the LLM as your background. Edit this cell to keep it current.

In [ ]:
#  Your Profile (used to personalize all analysis)

MY_PROFILE = """
Name: Muazzam Hussain
Education: 4th semester BS Artificial Intelligence, PAF-IAST, Karachi, Pakistan
Specialization: Agentic AI & RAG Systems
Seeking: Summer internship in AI/automation engineering

Projects:
- AI Voice Agent: Built an end-to-end voice agent using LLM orchestration and speech APIs
- n8n Content Automation Pipeline: Bilingual Urdu/English video editing agent using Deepgram, Ollama/Qwen, FFmpeg, and social media auto-posting
- House Price Prediction Model: ML regression model for real estate price estimation
- LinkedIn Post Generator: n8n workflow with Discord approval gate, Google Sheets, RSS feeds, and LinkedIn OAuth2

Technical Skills:
- Languages: Python
- AI/ML: LLM orchestration, RAG pipelines, Agentic AI, prompt engineering
- Tools: n8n, Deepgram, FFmpeg, Docker, Ollama, ComfyUI, HuggingFace
- APIs: OpenAI-compatible APIs, REST APIs
- Databases: SQL, basic vector DB concepts
- Other: BeautifulSoup, web scraping, workflow automation
"""

print("✓ Profile loaded")

## Web Scraper
Fetches and cleans a job posting from a URL

In [ ]:
# ── Job Posting Scraper ─

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class JobPosting:
    """
    Fetches a job posting URL and extracts clean text.
    Strips scripts, styles, nav, footers — keeps only the content.
    """
    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string.strip() if soup.title else "No title found"
        
        # Remove noise elements
        for tag in soup.body(["script", "style", "img", "input", "nav", "footer", "header"]):
            tag.decompose()
        
        self.text = soup.body.get_text(separator="\n", strip=True)
        
        # Truncate to avoid exceeding context window — 8000 chars is plenty for a job post
        if len(self.text) > 8000:
            self.text = self.text[:8000] + "\n\n[Content truncated]"
    
    def __repr__(self):
        return f"JobPosting(title='{self.title}', chars={len(self.text)})"

print("✓ JobPosting class ready")

## Prompts
Two prompts: a system prompt that defines the analyzer's behavior, and a user prompt that injects the job text + your profile.

In [ ]:
# ── System Prompt 
# This is the control layer — defines exactly what the LLM should do and how.

SYSTEM_PROMPT = """You are a senior technical recruiter and career coach specializing in AI and software engineering roles.

When given a job description and a candidate profile, you produce a structured analysis in markdown with these exact sections:

## 1. Role Summary
One paragraph: what the company wants, what the role does, and the seniority level implied.

## 2. Required Skills
Bullet list of hard requirements — things the job description explicitly demands.

## 3. Nice-to-Have Skills
Bullet list of preferred or bonus skills mentioned.

## 4. Red Flags
Honest assessment: anything concerning about this role (vague scope, unrealistic expectations, skills the candidate clearly lacks, signs of a poor environment). Be direct — do not sugarcoat.

## 5. Candidate Fit Score
Score out of 10 with a one-paragraph justification. Be accurate, not encouraging.

## 6. Skill Gaps
Specific skills the candidate is missing that this role requires. List only real gaps — not things the candidate has.

## 7. Your Strongest Selling Points for This Role
From the candidate's profile, the 3-5 things most directly relevant to THIS specific job. Be concrete.

## 8. Tailored Application Pitch (3–4 sentences)
A punchy, specific pitch paragraph the candidate can adapt for their cover letter or application form. 
Reference the company and role by name. Lead with impact, not background.

## 9. Interview Prep Questions
5 likely technical or behavioral questions this company will ask, based on the job description.
For each question, add a one-line hint on how to answer it using the candidate's actual experience.

Be precise. Do not pad. If something is unclear from the job description, say so."""


def build_user_prompt(job: JobPosting) -> str:
    """Constructs the user prompt by combining the job text with the candidate profile."""
    return f"""Analyze this job posting and match it against the candidate profile below.

## Job Posting URL
{job.url}

## Job Posting Content
{job.text}

---

## Candidate Profile
{MY_PROFILE}
"""

print("✓ Prompts defined")

## Core Analysis Function
Ties everything together — scrape → prompt → API call → display.

In [ ]:
# ── Analyzer ──────────────────────────────────────────────────────────────────

def analyze_job(url: str) -> str:
    """
    Takes a job posting URL.
    Returns a full markdown analysis personalized to Muazzam's profile.
    """
    print(f"🔍 Scraping: {url}")
    job = JobPosting(url)
    print(f"✓ Scraped: '{job.title}' ({len(job.text)} chars)")
    
    print("🤖 Analyzing with LLM...")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(job)}
        ]
    )
    
    return response.choices[0].message.content


def display_analysis(url: str):
    """Runs analyze_job and renders the result as formatted markdown in Jupyter."""
    result = analyze_job(url)
    print("\n" + "─"*60 + "\n")
    display(Markdown(result))


print("✓ Analyzer ready — run the cell below to start")

## Run 
Paste any job posting URL below and run.

In [ ]:
# ── Paste your job URL here and run ──────────────────────────────────────────

JOB_URL = "https://www.linkedin.com/jobs/view/4418757855"  # ← change this

display_analysis(JOB_URL)

## Batch Mode — Analyze Multiple Jobs at Once
Paste a list of URLs and get a ranked comparison.

In [ ]:
# ── Batch Analysis ────────────────────────────────────────────────────────────
# Useful when you're applying to multiple roles and want to prioritize.

JOB_URLS = [
    "https://example.com/job1",  # ← replace with real URLs
    "https://example.com/job2",
    "https://example.com/job3",
]

for i, url in enumerate(JOB_URLS, 1):
    display(Markdown(f"---\n# Job {i} of {len(JOB_URLS)}"))
    display_analysis(url)